# 07 — CLI Example

Demonstrates every `beamlab21` subcommand by calling the CLI via `subprocess`.  
All data is synthesised inline — no real beam cube or CST export is needed.

| Step | Command |
|---|---|
| 0 | `beamlab21 --version` |
| 1–2 | Synthesise beam cube + write YAML config |
| 3 | `beamlab21 fit` — single-channel Gaussian + Zernike |
| 4 | `beamlab21 fit-all` — all channels, stacks CSVs |
| 5 | `beamlab21 compute` — regenerate Zernike model from coefficients |
| 6 | Synthesise CST `.txt` exports |
| 7 | `beamlab21 cst --skip-fit` — convert one CST file to `.npz` |
| 8 | `beamlab21 cst-stack` — merge three frequencies |
| 9 | `beamlab21 fit-all --datafile` — fit the stacked CST cube |

**Requirements:** `pip install -e ".[dev]"`

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

import numpy as np

from beamlab21.models import twoD_Gaussian
from beamlab21.zernike import zernike_mode

# All outputs land in a temporary directory
scratch = Path(tempfile.mkdtemp())
outputs     = scratch / "outputs"
outputs_all = scratch / "outputs_all"
print(f"Scratch: {scratch}")

def run(args):
    """Run a CLI command, stream output, raise on failure."""
    print("\n$", " ".join(str(a) for a in args))
    result = subprocess.run(args, check=True)

## Step 0 · Version

In [ ]:
run(["beamlab21", "--version"])

## Step 1 · Synthesise a single-frequency beam cube

In [ ]:
N_pix, xy_max = 201, 30.0
x = np.linspace(-xy_max, xy_max, N_pix)
y = np.linspace(-xy_max, xy_max, N_pix)
X, Y = np.meshgrid(x, y)

params = [1.0, 9.0, 8.0, 0.3, -0.2, 5.0]   # amp, sigx, sigy, xo, yo, tilt
beam = twoD_Gaussian(x, y, params)

# Small coma perturbation (Zernike Z7)
xm = X / params[1]; ym = Y / params[2]
rm = np.hypot(xm, ym); rm[rm == 0] = 1e-10
beam += 0.04 * zernike_mode(3, 1, rm, np.arctan2(ym, xm))
beam = np.clip(beam / beam.max(), 0, None)

cube_400 = scratch / "beam_400.npz"
np.savez(cube_400, x=x, y=y, freq=np.array([0.4]), data=beam[np.newaxis])
print(f"Cube: {cube_400}  shape={beam[np.newaxis].shape}")

## Step 2 · Write config files

In [ ]:
def fit_config(data_dir, datafile, out_dir, save_models=True):
    return textwrap.dedent(f"""
        Telescope_name: "Synthetic"
        frequency: 400
        data_dir:  "{data_dir}"
        datafile:  "{datafile}"
        coord_type: cartesian
        normalize_data: True
        gaussian_error_type: uniform
        init_gparams: [1.0, 10.0, 10.0, 0.0, 0.0, 0.0]
        gminimize_method: Nelder-Mead
        gtol: 1.0e-6
        gverbose: False
        save_gaussian_model: {str(save_models).capitalize()}
        goutput_dir:  "{out_dir}"
        goutput_name: "GaussianModel_{{{{ frequency }}}}"
        goutput_format: ".npz"
        zernike_error_type: proportional
        N: 10
        skip_minimise: True
        fac: 3
        ztminimize_method: Nelder-Mead
        ztmaxiter: 10
        zttol: 1.0e-6
        zverbose: False
        save_zernike_model: {str(save_models).capitalize()}
        zoutput_dir:  "{out_dir}"
        zoutput_name: "ZernikeModel_{{{{ frequency }}}}"
        zoutput_format: ".npz"
        plot_results: False
        plot_cmap:    gist_ncar
        plot_format:  ".png"
        plot_directory: "{out_dir}"
        save_params:  True
        out_coef_dir: "{out_dir}"
        out_coef_name: "coefficients_{{{{ frequency }}}}.csv"
        out_sp_dir:   "{out_dir}"
        out_sp_name:  "scaleparameters_{{{{ frequency }}}}.csv"
    """).strip()

config_fit = scratch / "config_fit.yaml"
config_fit.write_text(fit_config(scratch, "beam_400.npz", outputs))

config_fitall = scratch / "config_fitall.yaml"
config_fitall.write_text(fit_config(scratch, "cube_multi.npz", outputs_all, save_models=False))

print("config_fit.yaml   →", config_fit)
print("config_fitall.yaml →", config_fitall)

## Step 3 · `beamlab21 fit`  — single-channel Gaussian + Zernike

In [ ]:
run(["beamlab21", "fit", str(config_fit)])
print("\nOutputs:", [f.name for f in sorted(outputs.iterdir())])

## Step 4 · `beamlab21 fit-all`  — all frequency channels

In [ ]:
freqs_mhz = [300.0, 400.0, 500.0]
slices = []
for f in freqs_mhz:
    scale = 400.0 / f
    b = twoD_Gaussian(x, y, [1.0, 9.0*scale, 8.0*scale, 0.3, -0.2, 5.0])
    slices.append(b / b.max())

cube_multi = scratch / "cube_multi.npz"
np.savez(cube_multi, x=x, y=y,
         freq=np.array(freqs_mhz) / 1e3,
         data=np.stack(slices, axis=0))
print(f"Multi-channel cube: {cube_multi}  shape={np.stack(slices).shape}")

run(["beamlab21", "fit-all", str(config_fitall)])
print("\nOutputs:", [f.name for f in sorted(outputs_all.iterdir())])

sp_all = outputs_all / "scaleparameters_all.csv"
if sp_all.exists():
    print("\nscaleparameters_all.csv (first 4 rows):")
    print(sp_all.read_text().splitlines()[0])
    for line in sp_all.read_text().splitlines()[1:4]:
        print(line)

## Step 5 · `beamlab21 compute`  — regenerate beam from coefficients

In [ ]:
config_compute = scratch / "config_compute.yaml"
config_compute.write_text(textwrap.dedent(f"""
    Telescope_name: "Synthetic"
    frequency: 400
    aperture_diameter: 6.0
    pixels: 128
    angular_res: 0.5
    dtype: float32
    coord_type: cartesian
    gen_gaussian_model: False
    gaussian_lamdabyD_coef: 1.03
    gaussian_amp: 1.0
    gaussian_offset_x: 0.0
    gaussian_offset_y: 0.0
    gaussian_rotation: 0.0
    add_noise2gaussian: False
    gaussian_noise_mean: 0.0
    gaussian_noise_sigma: 0.0
    gaussian_noise_random_seed: 42
    save_gaussian_model: False
    goutput_dir:  "{outputs}"
    goutput_name: "ComputedGaussian_{{{{ frequency }}}}"
    goutput_format: ".npz"
    coef_dir:       "{outputs}"
    coef_file:      "coefficients_{{{{ frequency }}}}.csv"
    scaleparam_dir: "{outputs}"
    scaleparam_file: "scaleparameters_{{{{ frequency }}}}.csv"
    gen_zernike_model: True
    add_noise2zernike: False
    zernike_noise_mean: 0.0
    zernike_noise_sigma: 0.0
    zernike_noise_random_seed: 42
    save_zernike_model: True
    zoutput_dir:  "{outputs}"
    zoutput_name: "ComputedZernike_{{{{ frequency }}}}"
    zoutput_format: ".npz"
    gen_airy_pattern: False
    airy_blockage_diameter: 0.0
    airy_offset_x: 0.0
    airy_offset_y: 0.0
    save_airy_pattern: False
    aoutput_dir:  "{outputs}"
    aoutput_name: "ComputedAiry_{{{{ frequency }}}}"
    aoutput_format: ".npz"
""").strip())

run(["beamlab21", "compute", str(config_compute)])

computed = list(outputs.glob("ComputedZernike*"))
print("\nComputed model:", [f.name for f in computed])

## Step 6 · Synthesise CST far-field text files

In [ ]:
def write_cst(path, freq_mhz, sigx=9.0, sigy=8.0):
    """Write a synthetic CST far-field .txt in the 8-column format CST exports."""
    thetas = np.linspace(0, 75, 76)
    phis   = np.linspace(0, 359, 360)
    T, P   = np.meshgrid(thetas, phis)
    T, P   = T.ravel(), P.ravel()
    px = T * np.cos(np.deg2rad(P))
    py = T * np.sin(np.deg2rad(P))
    amp    = np.exp(-(px**2 / (2*sigx**2) + py**2 / (2*sigy**2)))
    amp_dB = 20.0 * np.log10(np.clip(amp, 1e-3, None))
    data = np.column_stack([
        T, P, amp_dB, amp_dB - 20,
        np.zeros_like(T), amp_dB,
        np.zeros_like(T), np.zeros_like(T),
    ])
    header = (
        f"Synthetic CST export  freq={freq_mhz}MHz\n"
        "Theta Phi Abs(E) Abs(Cross) Phase(Cross) Abs(Copol) Phase(Copol) Ax.Ratio"
    )
    np.savetxt(path, data, header=header, comments="", fmt="%.6f")

cst_files = {}
for freq, sx, sy in [(400, 9.0, 8.0), (500, 7.2, 6.4), (600, 6.0, 5.3)]:
    p = scratch / f"farfield_{freq}.txt"
    write_cst(p, freq, sigx=sx, sigy=sy)
    cst_files[freq] = p

print("CST files:", [f.name for f in cst_files.values()])

## Step 7 · `beamlab21 cst`  — convert one CST export (skip fit)

In [ ]:
cst_cube_400 = scratch / "cst_cube_400.npz"

run([
    "beamlab21", "cst", str(cst_files[400]),
    "--freq",        "400",
    "--column",      "copol",
    "--size",        "201",
    "--xy-max",      "30.0",
    "--cube-output", str(cst_cube_400),
    "--config",      str(config_fit),
    "--skip-fit",
])

d = np.load(cst_cube_400)
print(f"\nConverted cube: shape={d['data'].shape}  freq={d['freq']*1e3} MHz")

## Step 8 · `beamlab21 cst-stack`  — merge three frequencies

In [ ]:
cst_cube_multi = scratch / "cst_cube_multi.npz"

run([
    "beamlab21", "cst-stack",
    str(cst_files[400]), str(cst_files[500]), str(cst_files[600]),
    "--freqs",  "400", "500", "600",
    "--output", str(cst_cube_multi),
    "--column", "copol",
    "--size",   "201",
    "--xy-max", "30.0",
])

d = np.load(cst_cube_multi)
print(f"\nStacked cube: shape={d['data'].shape}  freq={d['freq']*1e3} MHz")

## Step 9 · `beamlab21 fit-all`  — fit the stacked CST cube

In [ ]:
run([
    "beamlab21", "fit-all", str(config_fitall),
    "--datafile", str(cst_cube_multi),
    "--channels", "400", "500", "600",
])

print("\nOutputs:", [f.name for f in sorted(outputs_all.iterdir())])

## Clean up

In [ ]:
import shutil
shutil.rmtree(scratch, ignore_errors=True)
print(f"Scratch directory removed: {scratch}")